# Lighting Recommendation Transformer Training

This notebook trains a PyTorch Transformer regressor for the lighting recommendation model. It mounts Google Drive, reads `lightningData.csv`, builds recent per-room/per-lamp sequences, predicts an efficient recommended lamp level for the next five-minute recommendation step, and saves a zip file back to Drive.


## Optional GitHub Clone
This notebook is self-contained, but this cell also clones the project code so your Colab runtime follows the same GitHub + Drive workflow.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/VisualizationApp.git'  # <-- change this
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

# Change these if your Drive folder names are different.
DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATE_DATA_PATHS = [
    DRIVE_ROOT / 'VisualizationApp' / 'Data' / 'lightningData.csv',
    DRIVE_ROOT / 'VisualizationApp' / 'lightningData.csv',
    DRIVE_ROOT / 'lightningData.csv',
]

DATA_CSV = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), CANDIDATE_DATA_PATHS[0])
OUT_DIR = DRIVE_ROOT / 'VisualizationApp' / 'AIModelsAndAlgorithms' / 'LightingRecommendation' / 'transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_CSV:', DATA_CSV)
print('DATA_CSV exists:', DATA_CSV.exists())
print('OUT_DIR:', OUT_DIR)

if not DATA_CSV.exists():
    raise FileNotFoundError(
        'Could not find lightningData.csv in the default Drive locations. '
        'Set DATA_CSV manually to your file path, then rerun this cell.'
    )


DATA_CSV: /content/drive/MyDrive/VisualizationApp/lightningData.csv
DATA_CSV exists: True
OUT_DIR: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer


In [3]:
import json
import math
import random
import zipfile
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


device: cuda


In [4]:
# Training controls. Start smaller, then increase for long training.
MAX_ROWS = None          # Example for a quick run: 1_000_000. Use None for all rows.
MAX_SEQUENCES = 500_000  # Cap sequences to keep memory stable. Increase later if GPU/RAM allows.
SEQ_LEN = 12             # 12 samples = 1 hour if the data is every 5 minutes.
STRIDE = 3               # Use every 3rd possible sequence to reduce training size.
BATCH_SIZE = 512
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 96
N_HEADS = 4
N_LAYERS = 2
DIM_FEEDFORWARD = 192
DROPOUT = 0.15
PATIENCE = 4
LEVEL_MAX = 80.0
PREDICTION_HORIZON_MINUTES = 5


In [5]:
PERSONA_POLICY = {
    'StaticBright': {'factor': 0.90, 'cap': 75, 'min_on': 35},
    'Balanced': {'factor': 0.78, 'cap': 60, 'min_on': 25},
    'Routine': {'factor': 0.75, 'cap': 55, 'min_on': 25},
    'NightFocused': {'factor': 0.68, 'cap': 45, 'min_on': 15},
    'StaticDim': {'factor': 0.62, 'cap': 35, 'min_on': 10},
    'Housekeeping': {'factor': 1.00, 'cap': 80, 'min_on': 80},
    'Unknown': {'factor': 0.75, 'cap': 55, 'min_on': 20},
}

USECOLS = [
    'timestamp', 'room_number', 'floor', 'lamp_location', 'Value', 'room_state',
    'reservation_active', 'pir_motion', 'lightning_persona', 'n_occupants',
    'active_actors', 'hurry_morning', 'lazy_day', 'forgetful',
]

BASE_NUMERIC_COLS = [
    'value_scaled', 'pir_motion', 'n_occupants_scaled', 'active_actors_scaled',
    'hurry_morning', 'lazy_day', 'forgetful', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
]

CAT_COLS = ['lamp_location', 'reservation_active', 'occupancy_prediction', 'lighting_persona_prediction']

def cyclic(values, period):
    radians = 2 * np.pi * values / period
    return np.sin(radians), np.cos(radians)

def efficient_target(row):
    actual = float(row['Value'])
    occupancy = str(row['occupancy_prediction'])
    persona = str(row['lighting_persona_prediction'])
    policy = PERSONA_POLICY.get(persona, PERSONA_POLICY['Unknown'])

    if occupancy == 'Vacant' or row['reservation_active'] == 'No':
        return 0.0
    if occupancy == 'Cleaning' or persona == 'Housekeeping':
        return 80.0 if actual > 0 else 0.0
    if actual <= 0:
        return 0.0

    candidates = [
        row.get('lamp_value_mean_1h', np.nan),
        row.get('lamp_value_mean_3h', np.nan),
        row.get('room_value_mean_24h', np.nan),
    ]
    valid = [v for v in candidates if pd.notna(v)]
    recent = np.median(valid) if valid else np.nan
    if pd.isna(recent) or recent <= 0:
        recent = actual

    recommended = min(actual, recent, policy['cap']) * policy['factor']
    recommended = max(recommended, policy['min_on'])
    return float(np.clip(round(recommended), 0, 80))

def rolling_history(df, group_cols, value_col, window, output_col):
    return df.groupby(group_cols, sort=False)[value_col].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    ).rename(output_col)


In [6]:
def load_and_prepare(data_csv, max_rows=None):
    df = pd.read_csv(data_csv, usecols=USECOLS, parse_dates=['timestamp'], nrows=max_rows)
    df = df.sort_values(['room_number', 'lamp_location', 'timestamp']).reset_index(drop=True)

    df['Value'] = pd.to_numeric(df['Value'], errors='coerce').fillna(0).clip(0, 80)
    df['pir_motion'] = pd.to_numeric(df['pir_motion'], errors='coerce').fillna(0).clip(0, 1)
    for col in ['n_occupants', 'active_actors', 'hurry_morning', 'lazy_day', 'forgetful', 'floor']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['reservation_active'] = df['reservation_active'].fillna('Unknown').astype(str)
    df['lamp_location'] = df['lamp_location'].fillna('Unknown').astype(str)
    df['occupancy_prediction'] = df['room_state'].fillna('Unknown').astype(str)
    df['lighting_persona_prediction'] = df['lightning_persona'].fillna('Unknown').astype(str)

    df['is_on'] = df['Value'].gt(0).astype(float)
    df['hour_sin'], df['hour_cos'] = cyclic(df['timestamp'].dt.hour, 24)
    df['dow_sin'], df['dow_cos'] = cyclic(df['timestamp'].dt.dayofweek, 7)
    df['value_scaled'] = df['Value'] / LEVEL_MAX
    df['n_occupants_scaled'] = df['n_occupants'].clip(0, 8) / 8.0
    df['active_actors_scaled'] = df['active_actors'].clip(0, 8) / 8.0

    df['lamp_value_mean_1h'] = rolling_history(df, ['room_number', 'lamp_location'], 'Value', 12, 'lamp_value_mean_1h')
    df['lamp_value_mean_3h'] = rolling_history(df, ['room_number', 'lamp_location'], 'Value', 36, 'lamp_value_mean_3h')
    df['room_value_mean_24h'] = rolling_history(df, ['room_number'], 'Value', 288, 'room_value_mean_24h')
    df['recommended_level'] = df.apply(efficient_target, axis=1)

    for col in BASE_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('float32')
    df['target_scaled'] = (df['recommended_level'] / LEVEL_MAX).astype('float32')
    return df

df = load_and_prepare(DATA_CSV, max_rows=MAX_ROWS)
print('rows:', len(df))
print('time range:', df['timestamp'].min(), 'to', df['timestamp'].max())
df.head()


rows: 10563249
time range: 2022-01-06 00:00:00 to 2022-07-06 00:00:00


,timestamp,room_number,floor,lamp_location,Value,room_state,reservation_active,pir_motion,lightning_persona,n_occupants,...,dow_sin,dow_cos,value_scaled,n_occupants_scaled,active_actors_scaled,lamp_value_mean_1h,lamp_value_mean_3h,room_value_mean_24h,recommended_level,target_scaled
0,2022-01-22 08:45:00,1,1,bed_left,14,Occupied,Yes,1.0,StaticDim,2,...,-0.974928,-0.222521,0.175,0.25,0.25,NaN,NaN,NaN,10.0,0.125
1,2022-01-22 08:50:00,1,1,bed_left,14,Occupied,Yes,1.0,StaticDim,2,...,-0.974928,-0.222521,0.175,0.25,0.25,14.0,14.0,14.0,10.0,0.125
2,2022-01-22 08:55:00,1,1,bed_left,14,Occupied,Yes,1.0,StaticDim,2,...,-0.974928,-0.222521,0.175,0.25,0.25,14.0,14.0,14.0,10.0,0.125
3,2022-01-22 09:00:00,1,1,bed_left,14,Occupied,Yes,1.0,StaticDim,2,...,-0.974928,-0.222521,0.175,0.25,0.25,14.0,14.0,14.0,10.0,0.125
4,2022-01-22 09:05:00,1,1,bed_left,14,Occupied,Yes,1.0,StaticDim,2,...,-0.974928,-0.222521,0.175,0.25,0.25,14.0,14.0,14.0,10.0,0.125


In [7]:
category_values = {}
for col in CAT_COLS:
    values = sorted(df[col].fillna('Unknown').astype(str).unique().tolist())
    if 'Unknown' not in values:
        values.append('Unknown')
    category_values[col] = values

feature_names = list(BASE_NUMERIC_COLS)
for col, values in category_values.items():
    for value in values:
        feature_names.append(f'{col}={value}')

def encode_features(frame):
    parts = [frame[BASE_NUMERIC_COLS].to_numpy(dtype='float32')]
    for col, values in category_values.items():
        mapping = {value: i for i, value in enumerate(values)}
        ids = frame[col].fillna('Unknown').astype(str).map(mapping).fillna(mapping['Unknown']).astype(int).to_numpy()
        onehot = np.zeros((len(frame), len(values)), dtype='float32')
        onehot[np.arange(len(frame)), ids] = 1.0
        parts.append(onehot)
    return np.concatenate(parts, axis=1)

encoded = encode_features(df)
targets = df['target_scaled'].to_numpy(dtype='float32')
print('feature count:', encoded.shape[1])
print('categories:', {k: len(v) for k, v in category_values.items()})


feature count: 38
categories: {'lamp_location': 13, 'reservation_active': 3, 'occupancy_prediction': 4, 'lighting_persona_prediction': 7}


In [8]:
def build_sequences(frame, encoded_features, targets, seq_len, stride=1, max_sequences=None):
    sequences = []
    y = []
    end_indices = []

    for _, group in frame.groupby(['room_number', 'lamp_location'], sort=False):
        idx = group.index.to_numpy()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx), stride):
            window_idx = idx[end - seq_len + 1:end + 1]
            sequences.append(encoded_features[window_idx])
            y.append(targets[idx[end]])
            end_indices.append(idx[end])
            if max_sequences and len(sequences) >= max_sequences:
                x = np.stack(sequences).astype('float32')
                return x, np.array(y, dtype='float32'), np.array(end_indices, dtype=np.int64)

    if not sequences:
        raise ValueError('No sequences were built. Reduce SEQ_LEN or check your data.')
    x = np.stack(sequences).astype('float32')
    return x, np.array(y, dtype='float32'), np.array(end_indices, dtype=np.int64)

x, y, end_indices = build_sequences(df, encoded, targets, SEQ_LEN, STRIDE, MAX_SEQUENCES)
print('x:', x.shape, 'y:', y.shape)


x: (500000, 12, 38) y: (500000,)


In [9]:
# Chronological split by sequence end timestamp.
sequence_times = df.loc[end_indices, 'timestamp'].reset_index(drop=True)
order = np.argsort(sequence_times.to_numpy())
x = x[order]
y = y[order]
end_indices = end_indices[order]

n = len(x)
train_end = int(n * 0.80)
val_end = int(n * 0.90)

x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:val_end], y[train_end:val_end]
x_test, y_test = x[val_end:], y[val_end:]
test_indices = end_indices[val_end:]

print('train:', x_train.shape, 'val:', x_val.shape, 'test:', x_test.shape)
print('split times:', sequence_times.iloc[train_end], sequence_times.iloc[val_end])


train: (400000, 12, 38) val: (50000, 12, 38) test: (50000, 12, 38)
split times: 2022-04-13 21:25:00 2022-06-07 16:50:00


In [10]:
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)


In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class LightingRecommendationTransformer(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, n_layers, dim_feedforward, dropout):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.position = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.position(x)
        x = self.encoder(x)
        last_token = x[:, -1, :]
        return torch.sigmoid(self.head(last_token)).squeeze(-1)

model = LightingRecommendationTransformer(
    input_dim=x_train.shape[-1],
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
print(model)


/tmp/ipykernel_7325/911684877.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


LightingRecommendationTransformer(
  (input_projection): Linear(in_features=38, out_features=96, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
        )
        (linear1): Linear(in_features=96, out_features=192, bias=True)
        (dropout): Dropout(p=0.15, inplace=False)
        (linear2): Linear(in_features=192, out_features=96, bias=True)
        (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.15, inplace=False)
        (dropout2): Dropout(p=0.15, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=96, out_features=48, bias=True)
    (2): GE

In [12]:
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_count = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            pred = model(xb)
            loss = criterion(pred, yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * len(xb)
        total_count += len(xb)
    return total_loss / max(total_count, 1)

best_val = float('inf')
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f'epoch {epoch:02d} train_loss={train_loss:.5f} val_loss={val_loss:.5f}')

    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('early stopping')
            break

if best_state is not None:
    model.load_state_dict(best_state)


epoch 01 train_loss=0.00137 val_loss=0.00055
epoch 02 train_loss=0.00044 val_loss=0.00039
epoch 03 train_loss=0.00033 val_loss=0.00033
epoch 04 train_loss=0.00029 val_loss=0.00027
epoch 05 train_loss=0.00026 val_loss=0.00024
epoch 06 train_loss=0.00025 val_loss=0.00026
epoch 07 train_loss=0.00024 val_loss=0.00024
epoch 08 train_loss=0.00023 val_loss=0.00032
epoch 09 train_loss=0.00022 val_loss=0.00024
early stopping


In [13]:
@torch.no_grad()
def predict(loader):
    model.eval()
    preds = []
    actuals = []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        pred = model(xb).detach().cpu().numpy()
        preds.append(pred)
        actuals.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(actuals)

pred_scaled, actual_scaled = predict(test_loader)
pred_levels = np.clip(np.rint(pred_scaled * LEVEL_MAX), 0, LEVEL_MAX)
actual_levels = actual_scaled * LEVEL_MAX

mae = mean_absolute_error(actual_levels, pred_levels)
rmse = float(np.sqrt(mean_squared_error(actual_levels, pred_levels)))
r2 = r2_score(actual_levels, pred_levels)

test_rows = df.loc[test_indices].copy().reset_index(drop=True)
actual_energy_proxy = float(test_rows['Value'].sum())
target_energy_proxy = float(actual_levels.sum())
model_energy_proxy = float(pred_levels.sum())
target_saving_pct = 100 * (1 - target_energy_proxy / actual_energy_proxy) if actual_energy_proxy else 0
model_saving_pct = 100 * (1 - model_energy_proxy / actual_energy_proxy) if actual_energy_proxy else 0

print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)
print('model saving vs actual:', model_saving_pct)


MAE: 1.0265400409698486
RMSE: 1.880132960731093
R2: 0.9846152663230896
model saving vs actual: 29.179863201048008


In [14]:
model_path = OUT_DIR / 'lighting_recommendation_transformer.pt'
metadata_path = OUT_DIR / 'lighting_recommendation_transformer_metadata.json'
report_path = OUT_DIR / 'lighting_recommendation_transformer_report.txt'
predictions_path = OUT_DIR / 'lighting_recommendation_transformer_sample_predictions.csv'
zip_path = OUT_DIR / 'lighting_recommendation_transformer_outputs.zip'

metadata = {
    'model_type': 'transformer_regressor',
    'task': 'lighting_recommendation',
    'prediction_horizon_minutes': PREDICTION_HORIZON_MINUTES,
    'sequence_length': SEQ_LEN,
    'stride': STRIDE,
    'feature_names': feature_names,
    'base_numeric_cols': BASE_NUMERIC_COLS,
    'categorical_cols': CAT_COLS,
    'category_values': category_values,
    'level_min': 0,
    'level_max': LEVEL_MAX,
    'target': 'recommended_level',
    'input_dim': int(x_train.shape[-1]),
    'd_model': D_MODEL,
    'n_heads': N_HEADS,
    'n_layers': N_LAYERS,
    'dim_feedforward': DIM_FEEDFORWARD,
    'dropout': DROPOUT,
    'seed': SEED,
    'max_rows': MAX_ROWS,
    'max_sequences': MAX_SEQUENCES,
    'history': history,
    'metrics': {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)},
}

torch.save({'state_dict': model.state_dict(), 'metadata': metadata}, model_path)
metadata_path.write_text(json.dumps(metadata, indent=2))

sample = test_rows[[
    'timestamp', 'room_number', 'lamp_location', 'occupancy_prediction',
    'lighting_persona_prediction', 'reservation_active', 'Value',
]].copy()
sample['recommended_target'] = actual_levels
sample['model_prediction'] = pred_levels
sample.head(5000).to_csv(predictions_path, index=False)

report = '\n'.join([
    f'rows: {len(df):,}',
    f'sequences: {len(x):,}',
    f'train sequences: {len(x_train):,}',
    f'validation sequences: {len(x_val):,}',
    f'test sequences: {len(x_test):,}',
    f'sequence length: {SEQ_LEN}',
    f'prediction horizon minutes: {PREDICTION_HORIZON_MINUTES}',
    f'MAE: {mae:.3f}',
    f'RMSE: {rmse:.3f}',
    f'R2: {r2:.3f}',
    f'actual level-sum proxy: {actual_energy_proxy:.0f}',
    f'efficient target level-sum proxy: {target_energy_proxy:.0f}',
    f'model level-sum proxy: {model_energy_proxy:.0f}',
    f'target saving vs actual: {target_saving_pct:.2f}%',
    f'model saving vs actual: {model_saving_pct:.2f}%',
])
report_path.write_text(report)
print(report)


rows: 10,563,249
sequences: 500,000
train sequences: 400,000
validation sequences: 50,000
test sequences: 50,000
sequence length: 12
prediction horizon minutes: 5
MAE: 1.027
RMSE: 1.880
R2: 0.985
actual level-sum proxy: 2359521
efficient target level-sum proxy: 1676209
model level-sum proxy: 1671016
target saving vs actual: 28.96%
model saving vs actual: 29.18%


In [ ]:
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_path, metadata_path, report_path, predictions_path]:
        zf.write(path, arcname=path.name)

print('saved model:', model_path)
print('saved metadata:', metadata_path)
print('saved report:', report_path)
print('saved predictions:', predictions_path)
print('created zip:', zip_path)
print('You can download the zip from Google Drive:', zip_path)


saved model: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer/lighting_recommendation_transformer.pt
saved metadata: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer/lighting_recommendation_transformer_metadata.json
saved report: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer/lighting_recommendation_transformer_report.txt
saved predictions: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer/lighting_recommendation_transformer_sample_predictions.csv
created zip: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer/lighting_recommendation_transformer_outputs.zip
You can download the zip from Google Drive: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingRecommendation/transformer/lighting_recommendation_transformer_outputs.zip
